# ULTRON on Google Colab

Run the **full** Ultron app (dashboard, brain, tools, memory, SSE) on Colab's hardware instead of your own machine, reachable from your browser through a tunnel.

**What runs where**
- The FastAPI server, brain, tools, and dashboard run on the Colab VM.
- `run_shell` / `read_file` / `system_stats` act on the **Colab VM** (a throwaway Google container), not your computer.
- The LLM "brain" is still a cloud API (Groq here) — the same as running locally.

**Heads up: Colab sessions are temporary.** When the runtime resets, `ultron.db` and anything you didn't push to GitHub is gone. Cell 6 (optional) mounts Google Drive to persist the memory DB across sessions.

Run the cells top to bottom.

## 1. Clone (or update) the repo

In [ ]:
import os

REPO_URL = "https://github.com/Razoradams9/ultron.git"
REPO_DIR = "/content/ultron"

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 2. Install dependencies

The core app is lightweight. `pyngrok` is for the tunnel. Voice deps (Chatterbox/torch) are **not** installed here — do that in the optional voice cell only if you want spoken replies.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pyngrok nest_asyncio

## 3. Set your Groq API key

Paste your key from [console.groq.com/keys](https://console.groq.com/keys). It lives only in this session's memory — it is not written to the repo.

Optional: uncomment to switch persona to the polite butler.

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Paste your GROQ_API_KEY: ").strip()
os.environ["ULTRON_PROVIDER"] = "groq"
# os.environ["ULTRON_PERSONA"] = "jarvis"   # sardonic 'ultron' is the default

print("Key set." if os.environ["GROQ_API_KEY"] else "No key entered — the brain won't respond.")

## 4. (Optional) Get an ngrok token

A free [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) authtoken gives you a stable tunnel to the dashboard. Skip this cell to use Colab's built-in port proxy instead (Cell 5 handles both).

In [ ]:
from getpass import getpass

token = getpass("Paste your ngrok authtoken (or press Enter to skip): ").strip()
if token:
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    print("ngrok token set.")
else:
    print("Skipping ngrok — will use Colab's port proxy.")

## 5. Launch the server + open the dashboard

Starts uvicorn in the background on port 8000, then exposes it. Click the printed URL to open the ULTRON control room.

Re-run this cell if you edit code and want a fresh server.

In [ ]:
import threading, time, uvicorn, nest_asyncio

nest_asyncio.apply()  # Colab already runs an event loop; let uvicorn share it

PORT = 8000

def _serve():
    uvicorn.run("ultron.server:app", host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=_serve, daemon=True).start()
time.sleep(4)  # let it bind

public_url = None
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(PORT).public_url
    print("ULTRON dashboard:", public_url)
except Exception as e:
    print("ngrok unavailable (", e, ") — falling back to Colab proxy below.")
    try:
        from google.colab.output import eval_js
        print("ULTRON dashboard:", eval_js(f"google.colab.kernel.proxyPort({PORT})"))
    except Exception as e2:
        print("Colab proxy also unavailable:", e2)

## 6. (Optional) Persist memory across sessions

By default `ultron.db` is created in the repo folder and vanishes when the runtime resets. Run this cell **before Cell 5** to keep the conversation log + facts on your Google Drive instead.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.makedirs('/content/drive/MyDrive/ultron', exist_ok=True)
# # config.py reads ultron.db from the project root; symlink it to Drive:
# db = '/content/drive/MyDrive/ultron/ultron.db'
# link = '/content/ultron/ultron.db'
# if not os.path.islink(link):
#     if os.path.exists(link):
#         os.remove(link)
#     os.symlink(db, link)
# print('Memory persisted at', db)

## 7. (Optional) GPU-accelerated cloned voice

The voice service is the only heavy piece. Colab gives you a free GPU, so it runs far faster here than on CPU.

**First:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU*, then re-run cells 1–5.

This cell installs the voice deps, starts `voice/server.py` on port 8001 (it auto-detects CUDA via the new `VOICE_DEVICE` logic), and points the main app at it by setting `VOICE_URL`. Re-run Cell 5 afterward so the server picks up `VOICE_URL`.

In [ ]:
import os, threading, time

!pip install -q chatterbox-tts soundfile librosa

os.environ["VOICE_DEVICE"] = "cuda"   # force GPU; the service also auto-detects

def _serve_voice():
    import uvicorn
    uvicorn.run("voice.server:app", host="127.0.0.1", port=8001, log_level="warning")

threading.Thread(target=_serve_voice, daemon=True).start()
time.sleep(4)

os.environ["VOICE_URL"] = "http://127.0.0.1:8001"
print("Voice service starting on :8001 (GPU). Re-run Cell 5 so the app uses VOICE_URL.")
print("Model loads lazily on first synthesis — the first spoken reply takes a bit.")